# Qwen2-VL-2B-Instruct · Multimodal demo (HyperCube)

이 노트북은 HyperCube에 업로드된 Qwen2-VL-2B-Instruct 모델을 컨테이너 안에서 로드하고,
노트북 셀 안에 inline gradio UI를 띄워서 이미지+텍스트 입력을 받습니다.

전제: 이 컨테이너의 워크스페이스에 모델이 `/workspace/models/qwen2-vl-2b-instruct@v1/`
경로로 마운트되어 있고, 그 안에 `Qwen2-VL-2B-Instruct.tar.gz`가 있다고 가정합니다.

## 1. 의존성 설치 (한 번만)

In [ ]:
%pip install -q --upgrade transformers accelerate gradio pillow qwen-vl-utils

## 2. 모델 추출

tar.gz 형태로 업로드된 모델을 워크스페이스에 풉니다. 이미 풀려 있으면 skip.

In [ ]:
import os, tarfile, pathlib

MOUNT = pathlib.Path('/workspace/models/qwen2-vl-2b-instruct@v1')
DEST = pathlib.Path('/workspace/Qwen2-VL-2B-Instruct')

if DEST.exists() and (DEST / 'config.json').exists():
    print('already extracted:', DEST)
else:
    archives = list(MOUNT.glob('*.tar.gz')) + list(MOUNT.glob('*.tgz'))
    if not archives:
        raise SystemExit(f'no tar.gz found under {MOUNT}. Contents: {list(MOUNT.iterdir())}')
    archive = archives[0]
    print('extracting', archive, 'to', DEST.parent)
    with tarfile.open(archive) as t:
        t.extractall(DEST.parent)
    print('done. contents:', sorted(p.name for p in DEST.iterdir())[:10])

## 3. 모델 로드

FP16으로 GPU에 올립니다. 2B 파라미터 기준 약 5 GB VRAM 사용.

In [ ]:
import torch
from transformers import AutoProcessor, Qwen2VLForConditionalGeneration

MODEL_PATH = '/workspace/Qwen2-VL-2B-Instruct'

processor = AutoProcessor.from_pretrained(MODEL_PATH)
model = Qwen2VLForConditionalGeneration.from_pretrained(
    MODEL_PATH,
    torch_dtype=torch.float16,
    device_map='cuda',
)
model.eval()
print('loaded on', next(model.parameters()).device)

## 4. 추론 함수

In [ ]:
from PIL import Image

@torch.inference_mode()
def caption(image: Image.Image, prompt: str) -> str:
    if image is None:
        return '이미지를 업로드해주세요.'
    if not prompt or not prompt.strip():
        prompt = '이 이미지를 자세히 설명해주세요.'

    messages = [{
        'role': 'user',
        'content': [
            {'type': 'image'},
            {'type': 'text', 'text': prompt.strip()},
        ],
    }]
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = processor(text=[text], images=[image], return_tensors='pt').to('cuda')
    output_ids = model.generate(**inputs, max_new_tokens=256, do_sample=False)
    generated = output_ids[:, inputs.input_ids.shape[1]:]
    return processor.batch_decode(generated, skip_special_tokens=True)[0].strip()

## 5. Gradio UI (inline)

In [ ]:
import gradio as gr

demo = gr.Interface(
    fn=caption,
    inputs=[
        gr.Image(type='pil', label='이미지'),
        gr.Textbox(label='프롬프트 (비우면 기본 캡션)', placeholder='이 이미지를 자세히 설명해주세요.'),
    ],
    outputs=gr.Textbox(label='모델 응답', lines=8),
    title='Qwen2-VL-2B-Instruct',
    description='이미지 + 텍스트 프롬프트를 입력하면 모델이 응답합니다.',
    allow_flagging='never',
)
demo.launch(inline=True, share=False)